In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [3]:
from tqdm import tqdm
from pathlib import Path
from datetime import datetime

import numpy as np

In [4]:
import torch
import monai
from monai.utils import ensure_tuple_rep

from src.loader import get_dataloader
from src.utils import load_pretrain_model
from src.utils import same_seeds, load_config
from src.SlimUNETR.SlimUNETR import SlimUNETR
from monai.networks.nets import UNet

from main_unlab import calc_metrics_dict
from main_unlab import get_experiment_dir

from accelerate import Accelerator


In [5]:
device = torch.device('cuda:1')
torch.cuda.set_device(device)

In [6]:
config, data_flag, is_HepaticVessel = load_config()
# config.trainer.batch_size = 4
data_flag

'aneurysms'

In [14]:
same_seeds(config.trainer.seed)
image_size = config.trainer.image_size

model = SlimUNETR(**config.slim_unetr)
model.to(device)

train_loader, val_loader, unlab_loader = get_dataloader(config, data_flag)

brats2021 


['BraTS20_Training_001', 'BraTS20_Training_002', 'BraTS20_Training_003', 'BraTS20_Training_004', 'BraTS20_Training_005', 'BraTS20_Training_006', 'BraTS20_Training_007', 'BraTS20_Training_008', 'BraTS20_Training_009', 'BraTS20_Training_010', 'BraTS20_Training_011', 'BraTS20_Training_012', 'BraTS20_Training_013', 'BraTS20_Training_014', 'BraTS20_Training_015', 'BraTS20_Training_016', 'BraTS20_Training_017', 'BraTS20_Training_018', 'BraTS20_Training_019', 'BraTS20_Training_020', 'BraTS20_Training_021', 'BraTS20_Training_022', 'BraTS20_Training_023', 'BraTS20_Training_024', 'BraTS20_Training_025', 'BraTS20_Training_026', 'BraTS20_Training_027', 'BraTS20_Training_028', 'BraTS20_Training_029', 'BraTS20_Training_030', 'BraTS20_Training_031', 'BraTS20_Training_032', 'BraTS20_Training_033', 'BraTS20_Training_034', 'BraTS20_Training_035', 'BraTS20_Training_036', 'BraTS20_Training_037', 'BraTS20_Training_038', 'BraTS20_Training_039', 'BraTS20_Training_040', 'BraTS20_Training_041', 'B

d:\miniconda\envs\slim-unetr\Lib\site-packages\monai\utils\deprecate_utils.py:321: FutureWarning: monai.transforms.io.dictionary LoadImaged.__init__:image_only: Current default value of argument `image_only=False` has been deprecated since version 1.1. It will be changed to `image_only=True` in version 1.3.
  warn_deprecated(argname, msg, warning_category)


In [9]:
inference = monai.inferers.SlidingWindowInferer(
    roi_size=ensure_tuple_rep(config.trainer.image_size, dim=3),
    overlap=0.5,
    sw_device=device,
    device=device,
)

metrics = {
    "dice_metric": monai.metrics.DiceMetric(
        include_background=True,
        reduction=monai.utils.MetricReduction.MEAN_BATCH,
        get_not_nans=False,
    ),
    # 'hd95_metric': monai.metrics.HausdorffDistanceMetric(percentile=95, include_background=True, reduction=monai.utils.MetricReduction.MEAN_BATCH, get_not_nans=False)
}

post_trans = monai.transforms.Compose(
    [
        monai.transforms.Activations(sigmoid=True),
        monai.transforms.AsDiscrete(threshold=0.5),
    ]
)


In [9]:
logging_dir = Path(os.getcwd()) / "logs" / str(datetime.now()).replace(":", "_")
accelerator = Accelerator(log_with=["tensorboard"], project_dir=logging_dir)
accelerator.init_trackers("seed test")

In [10]:
base_exp_path_save = get_experiment_dir(config, data_flag, root="model_store")

In [11]:
root_path = Path('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/')

In [12]:
def replace_list(seed_list, path):
    for seed in seed_list:
        path = path.replace(seed + '\\', '')
    
    return path

In [13]:
all_list = list(root_path.rglob("*/seed*/epoch_799"))
exp_list_act = [str(exp) for exp in all_list]

In [14]:
exp_list_with_seed = []
for exp in set(map(lambda p: p.parent.parent, all_list)):
    exp_list_with_seed.append(list( p / "epoch_799" for p in exp.iterdir()))

In [15]:
exp_list_with_seed

[[WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_/seed25/epoch_799'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_/seed32/epoch_799'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_/seed42/epoch_799'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_/seed50/epoch_799')],
 [WindowsPa

In [15]:
def eval_model(pretrain_path, model):
    model = load_pretrain_model(pretrain_path, model, verbose=False)
    return model.eval()

In [16]:
def evaluate_experiment(model, exp_lists, device):
    

    metrics_value = []
    for exp_path in tqdm(exp_lists):
        checkpoint = str(exp_path / "pytorch_model.bin")
        # print(checkpoint)
        model = load_pretrain_model(checkpoint, model, verbose=False)
        model.eval()
        
        for image_batch in val_loader:
            logits = inference(image_batch["image"].to(device), model)
            val_outputs = [post_trans(i) for i in logits]
            for metric_name in metrics:
                metrics[metric_name](y_pred=val_outputs, y=image_batch["label"].to(device))

        _, batch_acc = calc_metrics_dict(
        metrics, accelerator, data_flag, is_train=False
        )

        metrics_value.append(batch_acc.cpu().numpy())

    return metrics_value # np.mean(metrics_value, axis=0), np.std(metrics_value, axis=0)

In [17]:
# np.mean(metrics_value, axis=0), np.std(metrics_value, axis=0)

In [18]:
# evaluate_experiment(model, exp_lists, device)

In [19]:
exp_list_with_seed

[[WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_/seed25/epoch_799'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_/seed32/epoch_799'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_/seed42/epoch_799'),
  WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_/seed50/epoch_799')],
 [WindowsPa

In [20]:
exp_eval_list = [
    [exp_lists[0].parent.parent, evaluate_experiment(model, exp_lists, device)]
    for exp_lists in tqdm(exp_list_with_seed)
]

100%|██████████| 8/8 [03:49<00:00, 28.70s/it]


In [22]:
exp_eval_list

[[WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_'),
  [array([0.86159   , 0.75541604, 0.61350256], dtype=float32),
   array([0.8691613 , 0.76333123, 0.6103376 ], dtype=float32),
   array([0.8676465, 0.7636749, 0.6006278], dtype=float32),
   array([0.86612624, 0.7707411 , 0.609515  ], dtype=float32)]],
 [WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfTrue/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/ps_post_tf/unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch200'),
  [array([0.8736496 , 0.77329457, 0.6014775 ], dtype=float32),
   array([0.8708412 , 0.78938156, 0.62560785], dtype=float32),
   array([0.8767991 , 0.79236615, 0.6519681 ], dtype=float32),
   array([0.8739646 , 0.78174525, 0.6560938 ], dtype

In [27]:
metr = [
    [
        exp_eval_list[i][0],
        (np.mean(exp_eval_list[i][1], axis=0), np.std(exp_eval_list[i][1], axis=0))
    ]
    for i in range(len(exp_eval_list))
]

In [29]:
exp_eval_list[6]

[WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/ps_post_tf/unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch100'),
 [array([0.8547615 , 0.7634348 , 0.58648694], dtype=float32),
  array([0.83625937, 0.7535227 , 0.61377275], dtype=float32),
  array([0.86001855, 0.78773934, 0.6372474 ], dtype=float32),
  array([0.8500323, 0.7496106, 0.5995204], dtype=float32)]]

In [28]:
metr

[[WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_'),
  (array([0.866131 , 0.7632909, 0.6084958], dtype=float32),
   array([0.00283284, 0.00542375, 0.00478028], dtype=float32))],
 [WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfTrue/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/ps_post_tf/unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch200'),
  (array([0.8738136, 0.7841969, 0.6337868], dtype=float32),
   array([0.00210939, 0.00739068, 0.02201674], dtype=float32))],
 [WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/unlab_ratio0.5_unlab_weight0.

In [ ]:
exp_eval_list = {
    str(exp_lists[0].parent.parent): evaluate_experiment(model, exp_lists, device)
    for exp_lists in tqdm(exp_list_with_seed)
}

  0%|          | 0/7 [00:00<?, ?it/s]

d:\sandbox\medical\Slim-UNETR\model_store\imageTBAD_MSD_preproc_128_border_crop64\epoch800\use_tfTrue\ims_128\batch_size_4\rot_prob0.7_rot_angle0.6283\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\ps_post_tf\unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch200\seed25\epoch_799\pytorch_model.bin


d:\sandbox\medical\Slim-UNETR\model_store\imageTBAD_MSD_preproc_128_border_crop64\epoch800\use_tfTrue\ims_128\batch_size_4\rot_prob0.7_rot_angle0.6283\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\ps_post_tf\unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch200\seed32\epoch_799\pytorch_model.bin


d:\sandbox\medical\Slim-UNETR\model_store\imageTBAD_MSD_preproc_128_border_crop64\epoch800\use_tfTrue\ims_128\batch_size_4\rot_prob0.7_rot_angle0.6283\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\ps_post_tf\unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch200\seed42\epoch_799\pytorch_model.bin


d:\sandbox\medical\Slim-UNETR\model_store\imageTBAD_MSD_preproc_128_border_crop64\epoch800\use_tfTrue\ims_128\batch_size_4\rot_prob0.7_rot_angle0.6283\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\ps_post_tf\unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch200\seed50\epoch_799\pytorch_model.bin


...

d:\sandbox\medical\Slim-UNETR\model_store\imageTBAD_MSD_preproc_128_border_crop64\epoch800\use_tfFalse\ims_128\batch_size_4\rot_prob0.7_rot_angle0.6283\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\only_labeled_\seed25\epoch_799\pytorch_model.bin


d:\sandbox\medical\Slim-UNETR\model_store\imageTBAD_MSD_preproc_128_border_crop64\epoch800\use_tfFalse\ims_128\batch_size_4\rot_prob0.7_rot_angle0.6283\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\only_labeled_\seed32\epoch_799\pytorch_model.bin


d:\sandbox\medical\Slim-UNETR\model_store\imageTBAD_MSD_preproc_128_border_crop64\epoch800\use_tfFalse\ims_128\batch_size_4\rot_prob0.7_rot_angle0.6283\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\only_labeled_\seed42\epoch_799\pytorch_model.bin


d:\sandbox\medical\Slim-UNETR\model_store\imageTBAD_MSD_preproc_128_border_crop64\epoch800\use_tfFalse\ims_128\batch_size_4\rot_prob0.7_rot_angle0.6283\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\only_labeled_\seed50\epoch_799\pytorch_model.bin


100%|██████████| 7/7 [03:01<00:00, 25.93s/it]


In [18]:
exp_eval_list

{'d:\\sandbox\\medical\\Slim-UNETR\\model_store\\imageTBAD_MSD_preproc_128_border_crop64\\epoch800\\use_tfTrue\\ims_128\\batch_size_4\\rot_prob0.7_rot_angle0.6283\\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\\ps_post_tf\\unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch200': (array([0.8738136 , 0.7841969 , 0.63378716], dtype=float32),
  array([0.00210939, 0.00739068, 0.02201624], dtype=float32)),
 'd:\\sandbox\\medical\\Slim-UNETR\\model_store\\imageTBAD_MSD_preproc_128_border_crop64\\epoch800\\use_tfFalse\\ims_128\\batch_size_4\\rot_prob0.7_rot_angle0.6283\\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\\unlab_ratio0.5_unlab_weight0.3_start_unlab_epoch1': (array([0.73087656, 0.6482948 , 0.44101834], dtype=float32),
  array([0., 0., 0.], dtype=float32)),
 'd:\\sandbox\\medical\\Slim-UNETR\\model_store\\imageTBAD_MSD_preproc_128_border_crop64\\epoch800\\use_tfFalse\\ims_128\\batch_size_4\\rot_prob0.7_rot_angle0.6283\\lrelu_split_new_class_GDFL_g2.0_fr08_fw080915\\ps_post_tf\\unlab_ratio

In [ ]:
 (array([0.85026795, 0.7635772 , 0.60925686], dtype=float32),
  array([0.00882556, 0.01483192, 0.0188224 ], dtype=float32)),

In [63]:
exp_eval_list

{WindowsPath('d:/sandbox/medical/Slim-UNETR/model_store/imageTBAD_MSD_preproc_128_border_crop64/epoch800/use_tfFalse/ims_128/batch_size_4/rot_prob0.7_rot_angle0.6283/lrelu_split_new_class_GDFL_g2.0_fr08_fw080915/only_labeled_'): (array([0.866131  , 0.76329064, 0.6084958 ], dtype=float32),
  array([0.00283284, 0.00542375, 0.00478028], dtype=float32))}